In [ ]:
# Mount Google Drive to access input data
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
# Install required libraries
# cartopy : coordinate reference system handling and map projections
!pip install cartopy


In [ ]:
# ============================================================
# Imports
# ============================================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.stats import mannwhitneyu
from itertools import combinations
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests


In [ ]:
# ============================================================
# Load and Preprocess Data
# ============================================================
# Loads the merged CRW / in-situ bleaching dataset, coerces
# numeric columns, drops incomplete rows, removes the invalid
# Bleaching_Alert sentinel value (-5), and creates a Site_Year
# identifier used as the random effect grouping variable.
# ============================================================

DATA_PATH = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/Scripts/Data/merged_data.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

# Coerce key columns to numeric; fill missing alert values with 0
df['Bleaching_Alert']   = pd.to_numeric(df['Bleaching_Alert'],   errors='coerce').fillna(0).astype(int)
df['Percent_Bleaching'] = pd.to_numeric(df['Percent_Bleaching'], errors='coerce')
df['Date_Year']         = pd.to_numeric(df['Date_Year'],         errors='coerce').astype(int)

# Drop rows missing any variable used in analysis
df_clean = df.dropna(subset=['Percent_Bleaching', 'Bleaching_Alert', 'Site_ID', 'Date_Year'])

# Remove sentinel value (-5) used for missing/invalid alert classifications
df_clean = df_clean[df_clean['Bleaching_Alert'] != -5].copy()

# Site-Year identifier: groups repeated measurements at the same site
# within the same year to account for spatial and temporal non-independence
df_clean['Site_Year'] = df_clean['Site_ID'].astype(str) + '_' + df_clean['Date_Year'].astype(str)

# Year-mean offset: used as a memory-safe fixed covariate in the crossed
# random effects model (see mixed-effects cell) to partition year-level
# variance without constructing a full crossed design matrix
year_means = df_clean.groupby('Date_Year')['Percent_Bleaching'].transform('mean')
df_clean['Year_Mean_Offset'] = year_means - year_means.mean()


In [ ]:
# ============================================================
# Figure 1: Global Survey Sites by CRW Bleaching Alert Level
# ============================================================
# Produces a 7-panel map showing in-situ survey locations
# coloured by their CRW Bleaching Alert level at time of survey
# (panels a–e), plus two diagnostic panels:
#   (f) Severe bleaching (>50%) with No Stress alert (false negatives)
#   (g) Severe bleaching (>50%) with Alert Level ≥3  (true positives)
# ============================================================

def plot_alert_level_maps(alert_df):
    alert_config = {
        0: {'label': 'No Stress',         'color': 'cyan'},
        1: {'label': 'Bleaching Watch',   'color': 'yellow'},
        2: {'label': 'Bleaching Warning', 'color': 'orange'},
        3: {'label': 'Alert Level 1',     'color': 'red'},
        4: {'label': 'Alert Level 2',     'color': 'darkred'},
    }

    # Diagnostic subsets for panels f and g
    severe_no_alert_df = alert_df[
        (alert_df['Percent_Bleaching'] > 50) & (alert_df['Bleaching_Alert'] == 0)
    ]
    severe_alerted_df = alert_df[
        (alert_df['Percent_Bleaching'] > 50) & (alert_df['Bleaching_Alert'] >= 3)
    ]

    extra_panels = [
        {'subset': severe_no_alert_df, 'color': 'deeppink',   'title': "Severe Bleaching (>50%) with No Stress Alert — False Negatives"},
        {'subset': severe_alerted_df,  'color': 'mediumblue', 'title': "Severe Bleaching (>50%) with Alert Level 1 or 2 — True Positives"},
    ]

    n_panels = 7
    fig = plt.figure(figsize=(15, 26))
    top, bottom, gap = 0.96, 0.02, 0.01
    h = (top - bottom - gap * (n_panels - 1)) / n_panels

    axes_list = []
    for panel_idx in range(n_panels):
        y0 = top - (panel_idx + 1) * h - panel_idx * gap
        ax = fig.add_axes([0.05, y0, 0.9, h], projection=ccrs.PlateCarree())
        axes_list.append(ax)

        ax.set_extent([-180, 180, -40, 45], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND,  facecolor='lightgray')
        ax.add_feature(cfeature.OCEAN, facecolor='aliceblue')
        ax.coastlines(linewidth=0.6)

        gl = ax.gridlines(draw_labels=True, linestyle='--', alpha=0.4, linewidth=0.5)
        gl.top_labels = gl.right_labels = False
        if panel_idx < n_panels - 1:
            gl.bottom_labels = False
        gl.xlabel_style = gl.ylabel_style = {'size': 14}

        if panel_idx < 5:
            cfg    = alert_config[panel_idx]
            subset = alert_df[alert_df['Bleaching_Alert'] == panel_idx]
            color, title = cfg['color'], f"{cfg['label']}  (n = {len(subset):,})"
        else:
            ep     = extra_panels[panel_idx - 5]
            subset = ep['subset']
            color, title = ep['color'], f"{ep['title']}  (n = {len(subset):,})"

        ax.scatter(
            subset['Longitude_Degrees'], subset['Latitude_Degrees'],
            color=color, s=15, edgecolor='black', linewidths=0.3,
            alpha=0.9, transform=ccrs.PlateCarree()
        )
        ax.set_title(title, fontsize=16, fontweight='bold', color='black', pad=3)
        for spine in ax.spines.values():
            spine.set_edgecolor('dimgray')
            spine.set_linewidth(0.8)
        ax.text(0.01, 0.97, f'({chr(97 + panel_idx)})',
                transform=ax.transAxes, fontsize=16, fontweight='bold',
                va='top', ha='left',
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.7))

    fig.suptitle(
        'NOAA CRW Bleaching Alert Levels & In Situ Survey Sites (1985–2020)',
        fontsize=16, fontweight='bold', y=0.98
    )
    fig.canvas.draw()
    for ax in axes_list:
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontsize(14)
    plt.show()


plot_alert_level_maps(df_clean)


In [ ]:
# ============================================================
# Mixed-Effects Models and Random Effects Structure Comparison
# ============================================================
# Four models are fit to compare random effects grouping strategies:
#
#   Model 1 — Time only       : (1 | Date_Year)
#   Model 2 — Site only       : (1 | Site_ID)
#   Model 3 — Site-Year       : (1 | Site_ID:Date_Year)
#   Model 4 — Site + Yr fixed : (1 | Site_ID) + Year_Mean_Offset
#
# Model 4 is a memory-safe approximation of crossed random effects.
# A full crossed model (Site + Year as separate random effects) would
# construct an 11,000+ column design matrix and crash the kernel.
# Instead, the year-level mean bleaching — centred on the grand mean —
# is entered as a fixed covariate, partitioning year-level variance
# without the memory cost.
#
# Models are compared via AIC, BIC, log-likelihood, and ICC.
# ICC = RE Variance / (RE Variance + Residual Variance)
# ============================================================

# --- Model 1: Time (Year) only ---
md_year     = smf.mixedlm("Percent_Bleaching ~ C(Bleaching_Alert)", df_clean, groups=df_clean["Date_Year"])
mdf_year    = md_year.fit()

# --- Model 2: Site only ---
md_site     = smf.mixedlm("Percent_Bleaching ~ C(Bleaching_Alert)", df_clean, groups=df_clean["Site_ID"])
mdf_site    = md_site.fit()

# --- Model 3: Site-Year interaction ---
md_siteyear  = smf.mixedlm("Percent_Bleaching ~ C(Bleaching_Alert)", df_clean, groups=df_clean["Site_Year"])
mdf_siteyear = md_siteyear.fit()

# --- Model 4: Site + year-mean offset (memory-safe crossed approximation) ---
md_crossed   = smf.mixedlm("Percent_Bleaching ~ C(Bleaching_Alert) + Year_Mean_Offset", df_clean, groups=df_clean["Site_ID"])
mdf_crossed  = md_crossed.fit()

# --- Model fit comparison table ---
models = {
    "Time Only      (1|Year)"               : mdf_year,
    "Site Only      (1|Site)"               : mdf_site,
    "Site-Year      (1|Site:Year)"          : mdf_siteyear,
    "Site + Yr offset (1|Site) + Yr fixed" : mdf_crossed,
}

fit_rows = []
for label, mdf in models.items():
    re_var = float(mdf.cov_re.iloc[0, 0]) if hasattr(mdf.cov_re, 'iloc') else np.nan
    fit_rows.append({
        "Model"      : label,
        "AIC"        : round(mdf.aic, 2),
        "BIC"        : round(mdf.bic, 2),
        "Log-Lik"    : round(mdf.llf, 2),
        "RE Variance": round(re_var, 3),
    })

fit_df = pd.DataFrame(fit_rows).set_index("Model")
print(fit_df.to_string())
print(f"\n* Lower AIC/BIC = better fit. Best AIC: {fit_df['AIC'].idxmin()}")

# --- Intraclass Correlation Coefficients ---
print("\nICC = RE Variance / (RE Variance + Residual Variance)")
for label, mdf in models.items():
    re_var  = float(mdf.cov_re.iloc[0, 0])
    res_var = mdf.scale
    icc     = re_var / (re_var + res_var)
    print(f"{label:<45s}  ICC = {icc:.3f}  (RE var={re_var:.2f}, Residual var={res_var:.2f})")

# --- Site + Year offset model: year fixed effect summary ---
re_var   = float(mdf_crossed.cov_re.iloc[0, 0])
res_var  = mdf_crossed.scale
total    = re_var + res_var
year_coef = mdf_crossed.params.get('Year_Mean_Offset', np.nan)
year_pval = mdf_crossed.pvalues.get('Year_Mean_Offset', np.nan)
print(f"\nSite + Year Offset model — year fixed effect:")
print(f"  Site ICC (after year adjustment): {re_var / total:.3f}")
print(f"  Year offset coefficient: {year_coef:.4f}  (p={year_pval:.4e})")

# Print full summary for the selected model (Site-Year)
print("\nFull summary — Site-Year model (used for pairwise comparisons):")
print(mdf_siteyear.summary())


In [ ]:
# ============================================================
# Pairwise Comparisons, Effect Sizes, Figure 2, and
# Supplementary Tables
# ============================================================
# 1. Mann-Whitney U tests for all 10 alert-level pairs with
#    Holm-Bonferroni multiple testing correction.
# 2. Rank-biserial correlation as effect size (r).
#    |r| < 0.10 = negligible, 0.10–0.30 = small,
#    0.30–0.50 = medium, > 0.50 = large
# 3. Figure 2: boxplot with significance brackets.
# 4. Supplementary CSV tables saved to Drive.
# ============================================================

ALERT_NAMES = {
    0: 'No Stress',
    1: 'Bleaching Watch',
    2: 'Bleaching Warning',
    3: 'Alert Level 1',
    4: 'Alert Level 2'
}
OUTPUT_DIR = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/Scripts/Data"

# ---- Helper functions ----

def rank_biserial(x, y):
    """
    Rank-biserial correlation: effect size for Mann-Whitney U test.
    Returns r in [-1, 1]; sign indicates direction of difference.
    """
    nx, ny = len(x), len(y)
    U, _   = mannwhitneyu(x, y, alternative='two-sided')
    return 1 - (2 * U) / (nx * ny)


def sig_symbol(p):
    """Convert a p-value to a significance symbol."""
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'


def effect_label(r):
    """Interpret rank-biserial correlation magnitude."""
    if abs(r) < 0.10: return 'negligible'
    if abs(r) < 0.30: return 'small'
    if abs(r) < 0.50: return 'medium'
    return 'large'


# ---- 1. Pairwise comparisons with Holm-Bonferroni correction ----

pairs = list(combinations([0, 1, 2, 3, 4], 2))
comparison_data = []

for a1, a2 in pairs:
    d1 = df_clean[df_clean['Bleaching_Alert'] == a1]['Percent_Bleaching']
    d2 = df_clean[df_clean['Bleaching_Alert'] == a2]['Percent_Bleaching']
    U, p_raw = mannwhitneyu(d1, d2, alternative='two-sided')
    r = rank_biserial(d1, d2)
    comparison_data.append({
        'Comparison'   : f"{ALERT_NAMES[a1]} vs {ALERT_NAMES[a2]}",
        'Alert_1'      : a1,
        'Alert_2'      : a2,
        'n1'           : len(d1),
        'n2'           : len(d2),
        'U_statistic'  : U,
        'p_value_raw'  : p_raw,
        'effect_size_r': r,
    })

comparison_df = pd.DataFrame(comparison_data)
reject, p_adj, _, _ = multipletests(comparison_df['p_value_raw'], alpha=0.05, method='holm')
comparison_df['p_value_adjusted'] = p_adj
comparison_df['significant']      = reject
comparison_df['significance']     = comparison_df['p_value_adjusted'].apply(sig_symbol)
comparison_df['effect_label']     = comparison_df['effect_size_r'].apply(effect_label)

comparison_df = comparison_df[[
    'Comparison', 'Alert_1', 'Alert_2', 'n1', 'n2',
    'U_statistic', 'p_value_raw', 'p_value_adjusted',
    'significance', 'effect_size_r', 'effect_label'
]]

# ---- 2. Figure 2: Boxplot with significance brackets ----

def plot_boxplot_with_significance(df, comparison_df):
    """
    Boxplot of Percent_Bleaching by Bleaching_Alert level with
    Holm-Bonferroni corrected significance brackets.
    Significant pairs are shown with solid black lines and symbols;
    non-significant pairs with faint dashed gray lines.
    """
    plot_df = df[(df['Bleaching_Alert'] != -5) & (df['Percent_Bleaching'].notna())].copy()
    plot_df['Bleaching_Alert_str'] = plot_df['Bleaching_Alert'].astype(str)

    n_counts   = plot_df.groupby('Bleaching_Alert')['Percent_Bleaching'].count()
    x_labels   = [f"{ALERT_NAMES[l]}\n($n={n_counts.get(l, 0)}$)" for l in range(5)]
    alert_colors = ['cyan', 'yellow', 'orange', 'red', 'darkred']

    plt.figure(figsize=(12, 8))
    ax = sns.boxplot(
        x='Bleaching_Alert_str', y='Percent_Bleaching', data=plot_df,
        order=[str(i) for i in range(5)],
        hue='Bleaching_Alert_str', hue_order=[str(i) for i in range(5)],
        palette=alert_colors, legend=False
    )
    plt.xticks(ticks=range(5), labels=x_labels)
    ax.axhline(y=50, color='black', linestyle='--', linewidth=2, alpha=0.7, zorder=1)

    y_base, y_step, highest_y = 105, 6, 0
    for i, row in comparison_df.reset_index(drop=True).iterrows():
        x1, x2 = row['Alert_1'], row['Alert_2']
        y, h   = y_base + i * y_step, 2
        sym    = row['significance']
        if sym != 'ns':
            plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
            plt.text((x1+x2)/2, y+h, sym, ha='center', va='bottom', color='k', fontsize=12)
        else:
            plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.0, c='gray', linestyle='--', alpha=0.3)
            plt.text((x1+x2)/2, y+h, 'ns', ha='center', va='bottom', color='gray', fontsize=10, alpha=0.5)
        highest_y = max(highest_y, y+h)

    plt.ylim(0, highest_y + 5)
    plt.yticks(np.arange(0, 101, 20))
    plt.xlabel('Bleaching Alert Level', fontsize=12)
    plt.ylabel('Percent Bleached (%)', fontsize=12)
    plt.title('Distribution of Percent Bleached Across Alert Levels', fontsize=13)
    plt.tight_layout()
    plt.show()


plot_boxplot_with_significance(df_clean, comparison_df)

# ---- 3. Supplementary Table 1: Pairwise comparisons ----

# Format for publication
fmt_df = comparison_df.copy()
fmt_df['U_statistic']   = fmt_df['U_statistic'].apply(lambda x: f"{x:.0f}")
fmt_df['p_value_raw']   = fmt_df['p_value_raw'].apply(lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}")
fmt_df['p_value_adjusted'] = fmt_df['p_value_adjusted'].apply(lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}")
fmt_df['effect_size_r'] = fmt_df['effect_size_r'].apply(lambda x: f"{x:.3f}")
print(fmt_df.to_string(index=False))

comparison_df.to_csv(f"{OUTPUT_DIR}/supplementary_table_pairwise_comparisons.csv", index=False)

# ---- 4. Supplementary Table 2: Descriptive statistics by alert level ----

summary_rows = []
for level in range(5):
    d = df_clean[df_clean['Bleaching_Alert'] == level]['Percent_Bleaching']
    summary_rows.append({
        'Alert_Level': level,
        'Alert_Name' : ALERT_NAMES[level],
        'n'          : len(d),
        'Mean'       : f"{d.mean():.2f}",
        'SD'         : f"{d.std():.2f}",
        'Median'     : f"{d.median():.2f}",
        'Q1'         : f"{d.quantile(0.25):.2f}",
        'Q3'         : f"{d.quantile(0.75):.2f}",
        'Min'        : f"{d.min():.2f}",
        'Max'        : f"{d.max():.2f}",
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

summary_df.to_csv(f"{OUTPUT_DIR}/supplementary_table_descriptive_statistics.csv", index=False)
